# Exercise 04 — MovieLens connectivity and resilience

This notebook extends Exercise 02 and Exercise 03. It analyzes the MovieLens bipartite graph using Lecture 04 concepts: connected components, articulation points, bridges, and network resilience through node and edge removal.

## Setup

Import the packages used for graph analysis and visualization.

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from copy import deepcopy

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
pd.options.display.max_rows = 20

## Load and rebuild the graph

Load `ratings.csv` and `movies.csv` and rebuild the same bipartite user–movie graph used in Exercises 02 and 03.

In [ ]:
ratings = pd.read_csv('../movielense/ml-latest-small/ratings.csv')
movies = pd.read_csv('../movielense/ml-latest-small/movies.csv')

ratings['user_node'] = ratings['userId'].astype(str).radd('user_')
ratings['movie_node'] = ratings['movieId'].astype(str).radd('movie_')

G = nx.Graph()
G.add_nodes_from([(u, {'bipartite': 0, 'node_type': 'user'}) for u in ratings['user_node'].unique()])
G.add_nodes_from([(m, {'bipartite': 1, 'node_type': 'movie'}) for m in ratings['movie_node'].unique()])
G.add_edges_from([(row.user_node, row.movie_node) for row in ratings.itertuples(index=False)])

print('Loaded graph with', G.number_of_nodes(), 'nodes and', G.number_of_edges(), 'edges')
print('Connected:', nx.is_connected(G))

## Connected components

Report the number and sizes of connected components in the original graph.

In [ ]:
components = list(nx.connected_components(G))
component_sizes = sorted([len(comp) for comp in components], reverse=True)

print(f'Number of connected components: {len(components)}')
print(f'Component sizes: {component_sizes}')
print(f'Largest component: {component_sizes[0]} nodes')
print(f'Percentage in largest: {100 * component_sizes[0] / G.number_of_nodes():.2f}%')

## Articulation points and bridges

Identify articulation points (cut vertices) and bridges (cut edges) in the graph. These are critical connectors whose removal would increase the number of connected components.

In [ ]:
articulation_points = list(nx.articulation_points(G))
bridges = list(nx.bridges(G))

print(f'Articulation points: {len(articulation_points)}')
print(f'Bridges: {len(bridges)}')

if articulation_points:
    print('\nSample articulation points (first 10):')
    for i, node in enumerate(articulation_points[:10]):
        node_type = G.nodes[node]['node_type']
        degree = G.degree(node)
        print(f'  {node} ({node_type}, degree={degree})')
else:
    print('No articulation points found.')

if bridges:
    print(f'\nSample bridges (first 10):')
    for i, (u, v) in enumerate(bridges[:10]):
        u_type = G.nodes[u]['node_type']
        v_type = G.nodes[v]['node_type']
        print(f'  {u} ({u_type}) -- {v} ({v_type})')
else:
    print('No bridges found.')

## Identify critical nodes for removal

Select one critical node: the highest-degree node that is also an articulation point (if one exists), otherwise the highest-degree node overall.

In [ ]:
# Find highest-degree articulation point, or just highest-degree node
ap_set = set(articulation_points)
node_degrees = dict(G.degree())

# Sort by degree
sorted_nodes = sorted(node_degrees.items(), key=lambda x: x[1], reverse=True)

# Prefer articulation point if available
critical_node = None
for node, degree in sorted_nodes:
    if node in ap_set:
        critical_node = node
        break

# Fallback to highest degree
if critical_node is None:
    critical_node = sorted_nodes[0][0]

critical_degree = G.degree(critical_node)
critical_type = G.nodes[critical_node]['node_type']
print(f'Critical node for removal: {critical_node}')
print(f'  Type: {critical_type}')
print(f'  Degree: {critical_degree}')
print(f'  Is articulation point: {critical_node in ap_set}')

## Remove critical node and measure impact

Remove the critical node and observe changes in connectivity.

In [ ]:
G_without_node = G.copy()
G_without_node.remove_node(critical_node)

components_after_node = list(nx.connected_components(G_without_node))
component_sizes_after_node = sorted([len(comp) for comp in components_after_node], reverse=True)

print(f'After removing node {critical_node}:')
print(f'  Number of components: {len(components_after_node)} (was {len(components)})')
print(f'  Component sizes: {component_sizes_after_node}')
print(f'  Largest component: {component_sizes_after_node[0]} nodes (was {component_sizes[0]})')
print(f'  Nodes lost from largest: {component_sizes[0] - component_sizes_after_node[0]}')
print(f'  Total nodes remaining: {G_without_node.number_of_nodes()}')

## Identify critical edge for removal

Select one critical edge: a bridge if any exist, otherwise the edge between the two highest-degree nodes.

In [ ]:
critical_edge = None

# Prefer a bridge
if bridges:
    critical_edge = bridges[0]
else:
    # Otherwise, find edge between high-degree nodes
    sorted_edges = sorted(G.edges(), key=lambda e: node_degrees[e[0]] + node_degrees[e[1]], reverse=True)
    critical_edge = sorted_edges[0]

u, v = critical_edge
u_type = G.nodes[u]['node_type']
v_type = G.nodes[v]['node_type']
u_degree = G.degree(u)
v_degree = G.degree(v)

print(f'Critical edge for removal: {u} -- {v}')
print(f'  {u} ({u_type}, degree={u_degree}) -- {v} ({v_type}, degree={v_degree})')
print(f'  Is bridge: {critical_edge in bridges}')

## Remove critical edge and measure impact

Remove the critical edge and observe changes in connectivity.

In [ ]:
G_without_edge = G.copy()
G_without_edge.remove_edge(u, v)

components_after_edge = list(nx.connected_components(G_without_edge))
component_sizes_after_edge = sorted([len(comp) for comp in components_after_edge], reverse=True)

print(f'After removing edge {u} -- {v}:')
print(f'  Number of components: {len(components_after_edge)} (was {len(components)})')
print(f'  Component sizes: {component_sizes_after_edge}')
print(f'  Largest component: {component_sizes_after_edge[0]} nodes (was {component_sizes[0]})')
print(f'  Nodes lost from largest: {component_sizes[0] - component_sizes_after_edge[0]}')
print(f'  Total edges remaining: {G_without_edge.number_of_edges()}')

## Summary table: Before and after removals

Compare connectivity metrics across the original graph, after node removal, and after edge removal.

In [ ]:
summary = pd.DataFrame({
    'Original': {
        'Nodes': G.number_of_nodes(),
        'Edges': G.number_of_edges(),
        'Components': len(components),
        'Largest Component': component_sizes[0],
        'Articulation Points': len(articulation_points),
        'Bridges': len(bridges)
    },
    'After Node Removal': {
        'Nodes': G_without_node.number_of_nodes(),
        'Edges': G_without_node.number_of_edges(),
        'Components': len(components_after_node),
        'Largest Component': component_sizes_after_node[0],
        'Articulation Points': len(list(nx.articulation_points(G_without_node))),
        'Bridges': len(list(nx.bridges(G_without_node)))
    },
    'After Edge Removal': {
        'Nodes': G_without_edge.number_of_nodes(),
        'Edges': G_without_edge.number_of_edges(),
        'Components': len(components_after_edge),
        'Largest Component': component_sizes_after_edge[0],
        'Articulation Points': len(list(nx.articulation_points(G_without_edge))),
        'Bridges': len(list(nx.bridges(G_without_edge)))
    }
})

summary

## Visualization: Subgraph around critical node

Draw a neighborhood around the critical node to visualize its role in connectivity.

In [ ]:
# Extract 2-hop neighborhood of critical node
neighbors_1 = set(G.neighbors(critical_node))
neighbors_2 = set()
for neighbor in neighbors_1:
    neighbors_2.update(G.neighbors(neighbor))

neighborhood = {critical_node} | neighbors_1 | neighbors_2
subG = G.subgraph(neighborhood).copy()

# Layout and visualization
pos = nx.spring_layout(subG, seed=42, k=2)

plt.figure(figsize=(12, 8))

# Draw edges
nx.draw_networkx_edges(subG, pos, alpha=0.6, width=1.5)

# Draw nodes: highlight critical node and by type
user_nodes = [n for n in subG if subG.nodes[n]['node_type'] == 'user']
movie_nodes = [n for n in subG if subG.nodes[n]['node_type'] == 'movie']

# Critical node in red, others by type
node_colors = []
node_sizes = []
for n in subG:
    if n == critical_node:
        node_colors.append('red')
        node_sizes.append(1500)
    elif n in user_nodes:
        node_colors.append('skyblue')
        node_sizes.append(800)
    else:
        node_colors.append('lightcoral')
        node_sizes.append(800)

nx.draw_networkx_nodes(subG, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9)

# Labels
labels = {n: n.replace('user_', 'U').replace('movie_', 'M') for n in subG}
nx.draw_networkx_labels(subG, pos, labels=labels, font_size=8)

plt.title(f'Neighborhood of critical node {critical_node} (red: critical node, blue: users, coral: movies)')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f'Neighborhood subgraph: {subG.number_of_nodes()} nodes, {subG.number_of_edges()} edges')